In [1]:
cd /workspace/gkuzmin/prompt_fairness/prompt-fairness/

/workspace/gkuzmin/prompt_fairness/prompt-fairness


In [2]:
import os, json, statistics
import pandas as pd
import src.utils.paper_artefacts as pa

# DEMOGRAPHY PROFILES

In [29]:
scorelists_by_model = {}
for model in pa.ALL_MODELS:
    model_specific_scores = []
    for annfilename in os.listdir(pa.FINAL_ANNOTATOR_DIR):
        annobj_for_df = {}
        annotator_filepath = os.path.join(pa.FINAL_ANNOTATOR_DIR, annfilename)
        with open(annotator_filepath, "r") as ao:
            annotator_object = json.load(ao)
        
        annobj_for_df["id"] = annotator_object["id"]
        annobj_for_df.update(**annotator_object["profile"])
        
        for i, (question, scores_for_question) in enumerate(annotator_object["answers"].items()):
            if not scores_for_question["answer"]:
                continue
            else:
                for qdi, scores in scores_for_question["downstream_scores"][model].items():
                    annobj_for_df[qdi] = list(scores.values())[0]
        model_specific_scores.append(annobj_for_df)
    score_df = pd.DataFrame(model_specific_scores)
    exclude_cols = ["id"] + pa.DEMOGRAPHY_PROFILES
    task_cols = [col for col in score_df.columns if col not in exclude_cols]

    fullscores_per_dp = {}
    for dp in pa.DEMOGRAPHY_PROFILES:
        fullscores_per_dp[dp] = {}
        for tc in task_cols:
            fullscores_per_dp[dp][tc] = {}
            if not any(tc.startswith(prefix) for prefix in pa.EXCLUDED_SCORES):
                demo_grouped = score_df.groupby([dp])
                for demo_name, group in demo_grouped:
                    # implement here for exceptions
                    if dp == "employment_sector":
                        group = group[group["language"] == "L1"]
                    elif dp == "education":
                        group = group[group["language"] == "L1"]
                    elif dp == "sex":
                        group = group[group["language"] == "L1"]
                        
                    elif(demo_name == "Black"):
                        group = group[group["language"] == "L1"]
                    elif(demo_name == "White"):
                        group = group[group["language"] == "L1"]
                    # implement here for exceptions
                    fullscores_per_dp[dp][tc][demo_name[0]] = group[tc].dropna().to_list()
    scorelists_by_model[model] = fullscores_per_dp

IsADirectoryError: [Errno 21] Is a directory: 'annotation_artefacts/final_annotator_objects_with_feats/.ipynb_checkpoints'

In [30]:
print(scorelists_by_model['gpt_4o'].keys())
print(scorelists_by_model['gpt_4o']['employment_sector'].keys())
print(scorelists_by_model['gpt_4o']['employment_sector']['0_0'].keys())
print(scorelists_by_model['gpt_4o']['employment_sector']['1_0'].keys())
print(scorelists_by_model['gpt_4o']['employment_sector']['1_0'].values())

KeyError: 'gpt_4o'

In [32]:
# sorry for this abomination, im too tired to think lmao

P_THRESHOLD = 0.05

divscores_model_v_taskg = {"Model" : pa.ALL_MODELS}
pairwise_ttest_model_v_taskg = {"Model" : pa.ALL_MODELS}
anova_model_v_taskg = {"Model" : pa.ALL_MODELS}
anova_corrected_model_v_taskg = {"Model" : pa.ALL_MODELS}

for tg_name, tg_idxs in pa.TASK_GROUP_DICT.items():
    divscore_obj_list = []
    pairwise_ttest_obj_list = []
    anova_obj_list = []
    anova_corrected_obj_list = []
    for modl in pa.ALL_MODELS:
        disvscore_table_row = {"model" : modl}
        pairwise_ttest_table_row = {"model" : modl}
        anova_table_row = {"model" : modl}
        anova_corrected_table_row = {"model" : modl}

        for k in scorelists_by_model[modl].keys():
            tasks = [qdi for qdi in scorelists_by_model[modl][k] if not any(qdi.startswith(prefix) for prefix in pa.EXCLUDED_SCORES)]
            tasks = [qdi for qdi in tasks if int(qdi.split("_")[0]) in tg_idxs]
            
            divsc_total = 0
            pairwise_t_test_significant = []
            anova_significant = []
            anova_corrected_significant = []
            
            for qdi in tasks:
                question_id = int(qdi.split("_")[0])
                clip_llm_judge = True if question_id in pa.LLM_JUDGE_TASKS else False
                qid_min = pa.INDEX_MM_LOOKUP_TABLE[question_id]["min"]
                qid_max = pa.INDEX_MM_LOOKUP_TABLE[question_id]["max"]

                divsc_total += pa.compute_diversity_score(scorelists_by_model[modl][k][qdi], qid_min, qid_max, clip_llm_judge)
                
                pairwise_ttest = pa.compute_pairwise_t_test(scorelists_by_model[modl][k][qdi], qid_min, qid_max, clip_llm_judge)
                if any(v < P_THRESHOLD for v in pairwise_ttest.values()):
                    pairwise_t_test_significant.append(qdi)
                
                anova_f, anova_p = pa.compute_anova_score(scorelists_by_model[modl][k][qdi], qid_min, qid_max, clip_llm_judge)
                if anova_p < P_THRESHOLD:
                    anova_significant.append(qdi)
                
                if anova_p * len(qdi) < P_THRESHOLD:
                    anova_corrected_significant.append(qdi)

            disvscore_table_row[k] = divsc_total/len(tasks)
            pairwise_ttest_table_row[k] = {
                "rejectable" : pairwise_t_test_significant,
                "all_tasks" : tasks,
            }
            anova_table_row[k] = {
                "rejectable" : anova_significant,
                "all_tasks" : tasks,
            }
            anova_corrected_table_row[k] = {
                "rejectable" : anova_corrected_significant,
                "all_tasks" : tasks,
            }
        
        divscore_obj_list.append(disvscore_table_row)
        pairwise_ttest_obj_list.append(pairwise_ttest_table_row)
        anova_obj_list.append(anova_table_row)
        anova_corrected_obj_list.append(anova_corrected_table_row)

    # ik bro but im paranoid
    assert(pa.ALL_MODELS == [m["model"] for m in divscore_obj_list])
    assert(pa.ALL_MODELS == [m["model"] for m in pairwise_ttest_obj_list])
    assert(pa.ALL_MODELS == [m["model"] for m in anova_obj_list])
    assert(pa.ALL_MODELS == [m["model"] for m in anova_corrected_obj_list])

    divscores_model_v_taskg[tg_name] = [statistics.mean([s for s in ls.values() if isinstance(s, float)]) for ls in divscore_obj_list]
    
    pairwise_ttest_model_v_taskg[tg_name] = []
    for ptt_by_model in pairwise_ttest_obj_list:
        all_rejectable = [v['rejectable'] if isinstance(v, dict) else [] for v in ptt_by_model.values()]
        all_rejectable = set([item for row in all_rejectable for item in row])
        pairwise_ttest_model_v_taskg[tg_name].append(len(all_rejectable)/len(ptt_by_model["employment_sector"]["all_tasks"]))
    
    anova_model_v_taskg[tg_name] = []
    for anova_by_model in anova_obj_list:
        all_rejectable = [v['rejectable'] if isinstance(v, dict) else [] for v in anova_by_model.values()]
        all_rejectable = set([item for row in all_rejectable for item in row])
        anova_model_v_taskg[tg_name].append(len(all_rejectable)/len(anova_by_model["employment_sector"]["all_tasks"]))
    
    anova_corrected_model_v_taskg[tg_name] = []
    for anova_corrected_by_model in anova_corrected_obj_list:
        all_rejectable = [v['rejectable'] if isinstance(v, dict) else [] for v in anova_corrected_by_model.values()]
        all_rejectable = set([item for row in all_rejectable for item in row])
        anova_corrected_model_v_taskg[tg_name].append(len(all_rejectable)/len(anova_corrected_by_model["employment_sector"]["all_tasks"]))

    def process_raw_obj(obj):
        return {k : (v if isinstance(v,str) else len(v["rejectable"])/len(v["all_tasks"])) for k, v in obj.items()}
    
    def convert_to_raw_str(input):
        if isinstance(input, str):
            return input
        elif isinstance(input, dict):
            return json.dumps(input)
    
    pairwise_ttest_obj_list_processed = [process_raw_obj(inst) for inst in pairwise_ttest_obj_list]
    anova_obj_list_processed = [process_raw_obj(inst) for inst in anova_obj_list]
    anova_corrected_obj_list_processed = [process_raw_obj(inst) for inst in anova_corrected_obj_list]
    
    pairwise_ttest_obj_list_rawstr = [{k : convert_to_raw_str(v) for k, v in dd.items()} for dd in pairwise_ttest_obj_list]
    anova_obj_list_rawstr = [{k : convert_to_raw_str(v) for k, v in dd.items()} for dd in anova_obj_list]
    anova_corrected_obj_list_rawstr = [{k : convert_to_raw_str(v) for k, v in dd.items()} for dd in anova_corrected_obj_list]

    pd.DataFrame(divscore_obj_list).to_csv(f"score_tables/divscores/divscores_{tg_name}.csv")

    pd.DataFrame(pairwise_ttest_obj_list_processed).to_csv(f"score_tables/ptt/ptt_{tg_name}.csv")
    pd.DataFrame(anova_obj_list_processed).to_csv(f"score_tables/anova/anova_{tg_name}.csv")
    pd.DataFrame(anova_corrected_obj_list_processed).to_csv(f"score_tables/anova_corrected/anova_corrected_{tg_name}.csv")

    pd.DataFrame(pairwise_ttest_obj_list_rawstr).to_csv(f"score_tables/ptt/rawstr/ptt_{tg_name}_rawstr.csv")
    pd.DataFrame(anova_obj_list_rawstr).to_csv(f"score_tables/anova/rawstr/anova_{tg_name}_rawstr.csv")
    pd.DataFrame(anova_corrected_obj_list_rawstr).to_csv(f"score_tables/anova_corrected/rawstr/anova_corrected_{tg_name}_rawstr.csv")

pd.DataFrame(divscores_model_v_taskg).to_csv(f"score_tables/divscores/divscores_model_v_taskg.csv")
pd.DataFrame(pairwise_ttest_model_v_taskg).to_csv(f"score_tables/ptt/ptt_model_v_taskg.csv")
pd.DataFrame(anova_model_v_taskg).to_csv(f"score_tables/anova/anova_model_v_taskg.csv")
pd.DataFrame(anova_corrected_model_v_taskg).to_csv(f"score_tables/anova_corrected/anova_corrected_model_v_taskg.csv")

/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:586: ConstantInputWarning: Each of the input arrays is constant; the F statistic is not defined or infinite
  res = hypotest_fun_out(*samples, **kwds)


# LINGUISTIC FEATURES

In [3]:
pa.ALL_TASKTYPE_DF_BY_MODEL.keys()
pa.TASK_TYPES

['machine_translation',
 'syntaxes',
 'knowledge_and_reasoning',
 'conditioned_text_gen',
 'qa_extraction',
 'summarization',
 'semantics']

In [4]:
pa.ALL_TASKTYPE_DF_BY_MODEL.keys()

dict_keys(['gpt_4o', 'llama31_70b_Inst', 'llama31_8B_Inst', 'llama32_1B_Inst', 'qwen25_72b_Inst', 'Qwen25_7B_Inst', 'Qwen25_1dot5B_Inst'])

In [5]:
#cols = list(taskdf.columns)
#cols

In [6]:
#task_lingcat_scores_by_model

In [7]:
#taskdf

In [5]:
import numpy as np
def encode_median(column):
    median = np.median(column)
    res = np.where(column <= median, 0, 1)
    #print(res)
    return res

In [13]:
pa.ALL_TASKTYPE_DF_BY_MODEL["gpt_4o"].keys()
pa.TASK_TYPES

['machine_translation',
 'syntaxes',
 'knowledge_and_reasoning',
 'conditioned_text_gen',
 'qa_extraction',
 'summarization',
 'semantics']

In [6]:
# NAIVE P VALUES
REJECT_THRESHOLD = 0.05
# 0 for TONE, 2 for GRAMMAR
CHAR_COLUMN = 5#0
DO_ENCODE_MEDIAN = True
char_cols = [4,5,6,7,8,9,10,11]
feats_names = [
    "num_words",
    "num_short_words",
    "num_chars",
    "num_stopwords",
    "num_nouns",
    "num_pronouns",
    "num_exclamation",
    "num_long_words",
]
# v2
char_cols = [0,2,4,5,6,7,8,9,10,11]
feats_names = [
    "sentence_type",
    "grammaticality",
    "ratio_exclamation",
    "ttr",
    "ratio_pronouns",
    "ratio_adjectives",
    "ratio_nouns",
    "num_words",
    "ratio_capital_words",
    "ratio_question",
]
# in case of additional linguistic feats COLS_STEP = 12
COLS_STEP = 12#4
for col_idx in range(len(char_cols)):
    CHAR_COLUMN = char_cols[col_idx]
    FEAT_NAME = feats_names[col_idx]
    print(FEAT_NAME, CHAR_COLUMN)
    
    task_lingcat_scores_by_model = {}
    for mmm in pa.ALL_TASKTYPE_DF_BY_MODEL.keys():
        task_lingcat_scores_by_model[mmm] = {}
        for tt in pa.TASK_TYPES:
            task_lingcat_scores_by_model[mmm][tt] = {}
            taskdf = pa.ALL_TASKTYPE_DF_BY_MODEL[mmm][tt].drop(columns=['id'])
            cols = list(taskdf.columns)
            #print(cols)
            #if cols[0].startswith("3_0"):
            #    print(taskdf)
            for i in range(0, len(cols), COLS_STEP):
                chunk = cols[i:i+COLS_STEP]
                task_qidx = chunk[-1].split("-")[-2]
                if not any(task_qidx.startswith(prefix) for prefix in pa.EXCLUDED_SCORES):
                    task_lingcat_scores_by_model[mmm][tt][task_qidx] = {}
                    chunkdf = taskdf[chunk].dropna()
                    print(chunkdf.columns)
                    if CHAR_COLUMN > 2 and DO_ENCODE_MEDIAN:
                        # encode numerical feat - binarize wrt median by task
                        #print(chunkdf[chunk[CHAR_COLUMN]])
                        chunkdf[chunk[CHAR_COLUMN]] = encode_median(chunkdf[chunk[CHAR_COLUMN]])
                        #print(chunkdf[chunk[CHAR_COLUMN]])
                    for ptone, tone_group in chunkdf.groupby(chunk[CHAR_COLUMN]):
                        task_lingcat_scores_by_model[mmm][tt][task_qidx][ptone] = tone_group[chunk[-1]].to_list()
    
    rejectable_objlist =[]
    for modelname in task_lingcat_scores_by_model.keys():
        row_instance = {"model" : modelname}
        for tg in task_lingcat_scores_by_model[modelname].keys():
            unfair_tasks = []
            for qidx in task_lingcat_scores_by_model[modelname][tg].keys():
                question_id = int(qidx.split("_")[0])
                clip_llm_judge = True if question_id in pa.LLM_JUDGE_TASKS else False
                qid_min = pa.INDEX_MM_LOOKUP_TABLE[question_id]["min"]
                qid_max = pa.INDEX_MM_LOOKUP_TABLE[question_id]["max"]
    
                if tg == "conditioned_text_gen":
                    print(modelname, tg)
                    print(task_lingcat_scores_by_model[modelname][tg][qidx])
                this_task_unfair = False
                ttest_result = pa.compute_pairwise_t_test(
                    task_lingcat_scores_by_model[modelname][tg][qidx],
                    qid_min,
                    qid_max,
                    clip_llm_judge
                )
                for ttest_value in ttest_result.values():
                    if ttest_value < REJECT_THRESHOLD:
                        this_task_unfair = True
                unfair_tasks.append(this_task_unfair)
            row_instance[tg] = sum(unfair_tasks)/len(unfair_tasks) 
        rejectable_objlist.append(row_instance)
    rejectable_df = pd.DataFrame(rejectable_objlist)
    sortdcols = ["model"] + sorted([c for c in rejectable_df.columns if c != "model"])
    rejectable_df = rejectable_df[sortdcols]
    
    if DO_ENCODE_MEDIAN:
        rejectable_df.to_csv(f"score_tables/p{FEAT_NAME}_ptt_median_feat_fixed_feats.csv")
    else:
        rejectable_df.to_csv(f"score_tables/p{FEAT_NAME}_ptt_fixed_feats.csv")

sentence_type 0
Index(['2_11-sentence_type', '2_11-sentence_length',
       '2_11-sentence_grammaticality', '2_11-ratio_exclamation', '2_11-ttr',
       '2_11-ratio_pronouns', '2_11-ratio_adjectives', '2_11-ratio_nouns',
       '2_11-num_words', '2_11-ratio_capital_words', '2_11-ratio_question',
       'Machine Translation-bug_Latn-2_11-bleu_score'],
      dtype='object')
Index(['2_3-sentence_type', '2_3-sentence_length',
       '2_3-sentence_grammaticality', '2_3-ratio_exclamation', '2_3-ttr',
       '2_3-ratio_pronouns', '2_3-ratio_adjectives', '2_3-ratio_nouns',
       '2_3-num_words', '2_3-ratio_capital_words', '2_3-ratio_question',
       'Machine Translation-deu_Latn-2_3-bleu_score'],
      dtype='object')
Index(['2_4-sentence_type', '2_4-sentence_length',
       '2_4-sentence_grammaticality', '2_4-ratio_exclamation', '2_4-ttr',
       '2_4-ratio_pronouns', '2_4-ratio_adjectives', '2_4-ratio_nouns',
       '2_4-num_words', '2_4-ratio_capital_words', '2_4-ratio_question',
       '

/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


gpt_4o conditioned_text_gen
{'declarative': [0.945, 0.9375, 0.9625, 0.9725], 'imperative': [0.96, 0.9675, 0.94, 0.9425, 0.9675, 0.955, 0.9475, 0.935, 0.9475, 0.945, 0.9575, 0.9675, 0.98, 0.9625, 0.965, 0.9525, 0.9625, 0.9375, 0.96, 0.935, 0.9475, 0.9475, 0.95, 0.9375, 0.935, 0.96, 0.94, 0.9475, 0.9575, 0.9525, 0.965, 0.94, 0.9375, 0.9425, 0.95, 0.9575, 0.9425, 0.9475, 0.94, 0.955, 0.955, 0.9475, 0.96, 0.95, 0.93, 0.9425, 0.9525, 0.9425, 0.9575, 0.9475, 0.9425, 0.96, 0.9625, 0.945, 0.94, 0.9575, 0.9625, 0.945, 0.955, 0.95, 0.9575, 0.9475, 0.945, 0.95, 0.9575, 0.96, 0.975, 0.955, 0.9475], 'interrogative': [0.955, 0.9625, 0.9475, 0.9625, 0.9625, 0.98]}
llama31_70b_Inst conditioned_text_gen
{'declarative': [0.96, 0.9675, 0.985, 0.98], 'imperative': [0.935, 0.96, 0.94, 0.9175, 0.94, 0.9675, 0.915, 0.8625, 0.9475, 0.9675, 0.94, 0.9275, 0.865, 0.94, 0.9175, 0.9525, 0.9675, 0.9075, 0.9525, 0.95, 0.96, 0.9575, 0.9425, 0.9575, 0.9725, 0.96, 0.88, 0.8925, 0.9375, 0.9475, 0.995, 0.9675, 0.89, 0.97

/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/_lib/deprecation.py:234: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return f(*args, **kwargs)


Qwen25_7B_Inst conditioned_text_gen
{'declarative': [0.26, 0.18, 0.25, 0.25], 'imperative': [0.09, 0.11, 0.28, 0.23, 0.03, 0.09, 0.05, 0.03, 0.19, 0.18, 0.26, 0.22, 0.23, 0.24, 0.28, 0.23, 0.15, 0.03, 0.16, 0.3, 0.25, 0.28, 0.29, 0.26, 0.24, 0.22, 0.07, 0.0, 0.16, 0.03, 0.38, 0.21, 0.04, 0.27, 0.29, 0.21, 0.16, 0.25, 0.27, 0.05, 0.04, 0.3, 0.23, 0.07, 0.24, 0.0, 0.02, 0.15, 0.24, 0.0, 0.21, 0.25, 0.12, 0.11, 0.07, 0.13, 0.26, 0.28, 0.04, 0.26, 0.01, 0.1, 0.06, 0.34, 0.29, 0.06, 0.03, 0.13, 0.22], 'interrogative': [0.14, 0.16, 0.05, 0.07, 0.08, 0.17]}
Qwen25_1dot5B_Inst conditioned_text_gen
{'declarative': [0.05, 0.03, 0.14, 0.05], 'imperative': [0.01, 0.01, 0.03, 0.04, 0.02, 0.05, 0.02, 0.0, 0.02, 0.0, 0.03, 0.04, 0.01, 0.04, 0.04, 0.02, 0.04, 0.01, 0.04, 0.02, 0.02, 0.05, 0.03, 0.01, 0.1, 0.03, 0.03, 0.01, 0.01, 0.04, 0.2, 0.02, 0.0, 0.05, 0.04, 0.04, 0.02, 0.04, 0.0, 0.06, 0.03, 0.04, 0.03, 0.02, 0.06, 0.01, 0.02, 0.06, 0.0, 0.02, 0.03, 0.04, 0.02, 0.01, 0.04, 0.02, 0.02, 0.05, 0.01,

/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


... (13 outputs left out)

/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


gpt_4o conditioned_text_gen
{0: [0.96, 0.9675, 0.94, 0.9425, 0.9675, 0.955, 0.9475, 0.945, 0.935, 0.9475, 0.945, 0.9575, 0.9675, 0.98, 0.9625, 0.965, 0.9525, 0.9625, 0.9375, 0.96, 0.935, 0.9475, 0.9475, 0.95, 0.9375, 0.96, 0.94, 0.9475, 0.9575, 0.9525, 0.965, 0.9375, 0.94, 0.9375, 0.9425, 0.95, 0.9575, 0.9425, 0.9625, 0.9475, 0.94, 0.9625, 0.955, 0.955, 0.9475, 0.96, 0.95, 0.93, 0.9425, 0.9525, 0.9425, 0.9575, 0.9475, 0.9425, 0.96, 0.9625, 0.945, 0.94, 0.9575, 0.9625, 0.945, 0.955, 0.95, 0.9575, 0.9475, 0.945, 0.95, 0.9575, 0.96, 0.975, 0.955, 0.9475, 0.9725], 1: [0.955, 0.935, 0.9625, 0.9475, 0.9625, 0.98]}
llama31_70b_Inst conditioned_text_gen
{0: [0.935, 0.96, 0.94, 0.9175, 0.94, 0.9675, 0.915, 0.96, 0.8625, 0.9475, 0.9675, 0.94, 0.9275, 0.865, 0.94, 0.9175, 0.9525, 0.9675, 0.9075, 0.9525, 0.95, 0.96, 0.9575, 0.9425, 0.9575, 0.96, 0.88, 0.8925, 0.9375, 0.9475, 0.995, 0.9675, 0.9675, 0.89, 0.975, 0.975, 0.9675, 0.97, 0.985, 0.9725, 0.96, 0.9825, 0.915, 0.9225, 0.9375, 0.9575, 0.9025,

/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/_lib/deprecation.py:234: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return f(*args, **kwargs)
/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


qwen25_72b_Inst conditioned_text_gen
{0: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 1: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
Qwen25_7B_Inst conditioned_text_gen
{0: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 1: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
Qwen25_1dot5B_Inst conditioned_text_gen
{0: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 1: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [22]:
# NOW THE SAME, BUT FOR ALL TASKS
# NAIVE P VALUES
REJECT_THRESHOLD = 0.05
# 0 for TONE, 2 for GRAMMAR
CHAR_COLUMN = 5#0
DO_ENCODE_MEDIAN = True
char_cols = [4,5,6,7,8,9,10,11]
feats_names = [
    "num_words",
    "num_short_words",
    "num_chars",
    "num_stopwords",
    "num_nouns",
    "num_pronouns",
    "num_exclamation",
    "num_long_words",
]
# v2
char_cols = [0,2,4,5,6,7,8,9,10,11]
feats_names = [
    "sentence_type",
    "grammaticality",
    "ratio_exclamation",
    "ttr",
    "ratio_pronouns",
    "ratio_adjectives",
    "ratio_nouns",
    "num_words",
    "ratio_capital_words",
    "ratio_question",
]
# in case of additional linguistic feats COLS_STEP = 12
COLS_STEP = 12#4
for col_idx in range(len(char_cols)):
    CHAR_COLUMN = char_cols[col_idx]
    FEAT_NAME = feats_names[col_idx]
    print(FEAT_NAME, CHAR_COLUMN)
    
    task_lingcat_scores_by_model = {}
    for mmm in pa.ALL_TASKTYPE_DF_BY_MODEL.keys():
        task_lingcat_scores_by_model[mmm] = {}
        for tt in pa.TASK_TYPES:
            task_lingcat_scores_by_model[mmm][tt] = {}
            taskdf = pa.ALL_TASKTYPE_DF_BY_MODEL[mmm][tt].drop(columns=['id'])
            cols = list(taskdf.columns)
            #print(cols)
            #if cols[0].startswith("3_0"):
            #    print(taskdf)
            for i in range(0, len(cols), COLS_STEP):
                chunk = cols[i:i+COLS_STEP]
                task_qidx = chunk[-1].split("-")[-2]
                if not any(task_qidx.startswith(prefix) for prefix in pa.EXCLUDED_SCORES):
                    task_lingcat_scores_by_model[mmm][tt][task_qidx] = {}
                    chunkdf = taskdf[chunk].dropna()
                    print(chunkdf.columns)
                    if CHAR_COLUMN > 2 and DO_ENCODE_MEDIAN:
                        # encode numerical feat - binarize wrt median by task
                        #print(chunkdf[chunk[CHAR_COLUMN]])
                        chunkdf[chunk[CHAR_COLUMN]] = encode_median(chunkdf[chunk[CHAR_COLUMN]])
                        #print(chunkdf[chunk[CHAR_COLUMN]])
                    for ptone, tone_group in chunkdf.groupby(chunk[CHAR_COLUMN]):
                        task_lingcat_scores_by_model[mmm][tt][task_qidx][ptone] = tone_group[chunk[-1]].to_list()
    
    rejectable_objlist =[]
    buf_res = {}
    for modelname in task_lingcat_scores_by_model.keys():
        row_instance = {"model" : modelname}
        buf_res[modelname] = {}
        tg = list(task_lingcat_scores_by_model[modelname].keys())[0]
        buf_res[modelname][tg] = {}
        for tg_idx in task_lingcat_scores_by_model[modelname].keys():
            #print(tg_idx)
            #print(task_lingcat_scores_by_model[modelname][tg_idx].keys())
            buf_res[modelname][tg].update(task_lingcat_scores_by_model[modelname][tg_idx])
        print(buf_res[modelname][tg].keys())
        unfair_tasks = []
        for qidx in buf_res[modelname][tg].keys():
            question_id = int(qidx.split("_")[0])
            clip_llm_judge = True if question_id in pa.LLM_JUDGE_TASKS else False
            qid_min = pa.INDEX_MM_LOOKUP_TABLE[question_id]["min"]
            qid_max = pa.INDEX_MM_LOOKUP_TABLE[question_id]["max"]

            if tg == "conditioned_text_gen":
                print(modelname, tg)
                #print(buf_res[modelname][tg][qidx])
            this_task_unfair = False
            ttest_result = pa.compute_pairwise_t_test(
                buf_res[modelname][tg][qidx],
                qid_min,
                qid_max,
                clip_llm_judge
            )
            for ttest_value in ttest_result.values():
                if ttest_value < REJECT_THRESHOLD:
                    this_task_unfair = True
            unfair_tasks.append(this_task_unfair)
        row_instance[tg] = sum(unfair_tasks)/len(unfair_tasks) 
        rejectable_objlist.append(row_instance)
    rejectable_df = pd.DataFrame(rejectable_objlist)
    sortdcols = ["model"] + sorted([c for c in rejectable_df.columns if c != "model"])
    rejectable_df = rejectable_df[sortdcols]
    
    if DO_ENCODE_MEDIAN:
        rejectable_df.to_csv(f"score_tables/all_p{FEAT_NAME}_ptt_median_feat_fixed_feats.csv")
    else:
        rejectable_df.to_csv(f"score_tables/all_p{FEAT_NAME}_ptt_fixed_feats.csv")

sentence_type 0
Index(['2_11-sentence_type', '2_11-sentence_length',
       '2_11-sentence_grammaticality', '2_11-ratio_exclamation', '2_11-ttr',
       '2_11-ratio_pronouns', '2_11-ratio_adjectives', '2_11-ratio_nouns',
       '2_11-num_words', '2_11-ratio_capital_words', '2_11-ratio_question',
       'Machine Translation-bug_Latn-2_11-bleu_score'],
      dtype='object')
Index(['2_3-sentence_type', '2_3-sentence_length',
       '2_3-sentence_grammaticality', '2_3-ratio_exclamation', '2_3-ttr',
       '2_3-ratio_pronouns', '2_3-ratio_adjectives', '2_3-ratio_nouns',
       '2_3-num_words', '2_3-ratio_capital_words', '2_3-ratio_question',
       'Machine Translation-deu_Latn-2_3-bleu_score'],
      dtype='object')
Index(['2_4-sentence_type', '2_4-sentence_length',
       '2_4-sentence_grammaticality', '2_4-ratio_exclamation', '2_4-ttr',
       '2_4-ratio_pronouns', '2_4-ratio_adjectives', '2_4-ratio_nouns',
       '2_4-num_words', '2_4-ratio_capital_words', '2_4-ratio_question',
       '

/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


dict_keys(['2_14', '2_12', '2_11', '2_10', '2_9', '2_8', '2_7', '2_6', '2_5', '2_4', '2_3', '2_2', '2_1', '2_0', '16_0', '17_0', '26_1', '26_0', '5_0', '6_0', '7_0', '8_0', '9_0', '10_0', '11_0', '12_0', '13_0', '14_0', '15_0', '18_0', '19_0', '28_0', '21_0', '22_0', '20_0', '3_0', '0_0', '1_0', '4_0'])
dict_keys(['2_6', '2_1', '2_8', '2_12', '2_9', '2_0', '2_7', '2_14', '2_10', '2_4', '2_3', '2_11', '2_2', '2_5', '16_0', '17_0', '26_1', '26_0', '5_0', '6_0', '7_0', '8_0', '9_0', '10_0', '11_0', '12_0', '13_0', '14_0', '15_0', '18_0', '19_0', '28_0', '21_0', '22_0', '20_0', '3_0', '0_0', '1_0', '4_0'])
dict_keys(['2_11', '2_2', '2_5', '2_10', '2_4', '2_3', '2_0', '2_7', '2_9', '2_14', '2_8', '2_6', '2_1', '2_12', '16_0', '17_0', '26_0', '26_1', '5_0', '6_0', '7_0', '8_0', '9_0', '10_0', '11_0', '12_0', '13_0', '14_0', '15_0', '18_0', '19_0', '28_0', '21_0', '22_0', '20_0', '3_0', '0_0', '1_0', '4_0'])
dict_keys(['2_14', '2_12', '2_11', '2_10', '2_9', '2_8', '2_7', '2_6', '2_5', '2_4', 

/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/_lib/deprecation.py:234: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return f(*args, **kwargs)


dict_keys(['2_6', '2_14', '2_1', '2_8', '2_9', '2_0', '2_7', '2_12', '2_4', '2_11', '2_3', '2_2', '2_5', '2_10', '16_0', '17_0', '26_0', '26_1', '5_0', '6_0', '7_0', '8_0', '9_0', '10_0', '11_0', '12_0', '13_0', '14_0', '15_0', '18_0', '19_0', '28_0', '21_0', '22_0', '20_0', '3_0', '0_0', '1_0', '4_0'])
dict_keys(['2_5', '2_11', '2_2', '2_3', '2_4', '2_10', '2_7', '2_14', '2_0', '2_9', '2_8', '2_1', '2_6', '2_12', '16_0', '17_0', '26_0', '26_1', '5_0', '6_0', '7_0', '8_0', '9_0', '10_0', '11_0', '12_0', '13_0', '14_0', '15_0', '18_0', '19_0', '28_0', '21_0', '22_0', '20_0', '3_0', '0_0', '1_0', '4_0'])
grammaticality 2
Index(['2_11-sentence_type', '2_11-sentence_length',
       '2_11-sentence_grammaticality', '2_11-ratio_exclamation', '2_11-ttr',
       '2_11-ratio_pronouns', '2_11-ratio_adjectives', '2_11-ratio_nouns',
       '2_11-num_words', '2_11-ratio_capital_words', '2_11-ratio_question',
       'Machine Translation-bug_Latn-2_11-bleu_score'],
      dtype='object')
Index(['2_3-se

/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


... (13 outputs left out)

/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


dict_keys(['2_11', '2_2', '2_5', '2_10', '2_4', '2_3', '2_0', '2_7', '2_9', '2_14', '2_8', '2_6', '2_1', '2_12', '16_0', '17_0', '26_0', '26_1', '5_0', '6_0', '7_0', '8_0', '9_0', '10_0', '11_0', '12_0', '13_0', '14_0', '15_0', '18_0', '19_0', '28_0', '21_0', '22_0', '20_0', '3_0', '0_0', '1_0', '4_0'])
dict_keys(['2_14', '2_12', '2_11', '2_10', '2_9', '2_8', '2_7', '2_6', '2_5', '2_4', '2_3', '2_2', '2_1', '2_0', '16_0', '17_0', '26_1', '26_0', '5_0', '6_0', '7_0', '8_0', '9_0', '10_0', '11_0', '12_0', '13_0', '14_0', '15_0', '18_0', '19_0', '28_0', '21_0', '22_0', '20_0', '3_0', '0_0', '1_0', '4_0'])
dict_keys(['2_6', '2_14', '2_1', '2_8', '2_9', '2_0', '2_7', '2_12', '2_4', '2_11', '2_3', '2_2', '2_5', '2_10', '16_0', '17_0', '26_0', '26_1', '5_0', '6_0', '7_0', '8_0', '9_0', '10_0', '11_0', '12_0', '13_0', '14_0', '15_0', '18_0', '19_0', '28_0', '21_0', '22_0', '20_0', '3_0', '0_0', '1_0', '4_0'])
dict_keys(['2_5', '2_11', '2_2', '2_3', '2_4', '2_10', '2_7', '2_14', '2_0', '2_9', '

/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/_lib/deprecation.py:234: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return f(*args, **kwargs)
/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


dict_keys(['2_6', '2_1', '2_8', '2_12', '2_9', '2_0', '2_7', '2_14', '2_10', '2_4', '2_3', '2_11', '2_2', '2_5', '16_0', '17_0', '26_1', '26_0', '5_0', '6_0', '7_0', '8_0', '9_0', '10_0', '11_0', '12_0', '13_0', '14_0', '15_0', '18_0', '19_0', '28_0', '21_0', '22_0', '20_0', '3_0', '0_0', '1_0', '4_0'])
dict_keys(['2_11', '2_2', '2_5', '2_10', '2_4', '2_3', '2_0', '2_7', '2_9', '2_14', '2_8', '2_6', '2_1', '2_12', '16_0', '17_0', '26_0', '26_1', '5_0', '6_0', '7_0', '8_0', '9_0', '10_0', '11_0', '12_0', '13_0', '14_0', '15_0', '18_0', '19_0', '28_0', '21_0', '22_0', '20_0', '3_0', '0_0', '1_0', '4_0'])
dict_keys(['2_14', '2_12', '2_11', '2_10', '2_9', '2_8', '2_7', '2_6', '2_5', '2_4', '2_3', '2_2', '2_1', '2_0', '16_0', '17_0', '26_1', '26_0', '5_0', '6_0', '7_0', '8_0', '9_0', '10_0', '11_0', '12_0', '13_0', '14_0', '15_0', '18_0', '19_0', '28_0', '21_0', '22_0', '20_0', '3_0', '0_0', '1_0', '4_0'])
dict_keys(['2_6', '2_14', '2_1', '2_8', '2_9', '2_0', '2_7', '2_12', '2_4', '2_11', '

In [14]:
# NAIVE P VALUES
REJECT_THRESHOLD = 0.05
# 0 for TONE, 2 for GRAMMAR
CHAR_COLUMN = 5#0
DO_ENCODE_MEDIAN = True
char_cols = [4,5,6,7,8,9,10,11]
feats_names = [
    "num_words",
    "num_short_words",
    "num_chars",
    "num_stopwords",
    "num_nouns",
    "num_pronouns",
    "num_exclamation",
    "num_long_words",
]
# in case of additional linguistic feats COLS_STEP = 12
COLS_STEP = 12#4
for col_idx in range(len(char_cols)):
    CHAR_COLUMN = char_cols[col_idx]
    FEAT_NAME = feats_names[col_idx]
    print(FEAT_NAME, CHAR_COLUMN)
    
    task_lingcat_scores_by_model = {}
    for mmm in pa.ALL_TASKTYPE_DF_BY_MODEL.keys():
        task_lingcat_scores_by_model[mmm] = {}
        for tt in pa.TASK_TYPES:
            task_lingcat_scores_by_model[mmm][tt] = {}
            taskdf = pa.ALL_TASKTYPE_DF_BY_MODEL[mmm][tt].drop(columns=['id'])
            cols = list(taskdf.columns)
            #print(cols)
            #if cols[0].startswith("3_0"):
            #    print(taskdf)
            for i in range(0, len(cols), COLS_STEP):
                chunk = cols[i:i+COLS_STEP]
                task_qidx = chunk[-1].split("-")[-2]
                if not any(task_qidx.startswith(prefix) for prefix in pa.EXCLUDED_SCORES):
                    task_lingcat_scores_by_model[mmm][tt][task_qidx] = {}
                    chunkdf = taskdf[chunk].dropna()
                    if CHAR_COLUMN > 2 and DO_ENCODE_MEDIAN:
                        # encode numerical feat - binarize wrt median by task
                        #print(chunkdf[chunk[CHAR_COLUMN]])
                        chunkdf[chunk[CHAR_COLUMN]] = encode_median(chunkdf[chunk[CHAR_COLUMN]])
                        #print(chunkdf[chunk[CHAR_COLUMN]])
                    for ptone, tone_group in chunkdf.groupby(chunk[CHAR_COLUMN]):
                        task_lingcat_scores_by_model[mmm][tt][task_qidx][ptone] = tone_group[chunk[-1]].to_list()
    
    rejectable_objlist =[]
    for modelname in task_lingcat_scores_by_model.keys():
        row_instance = {"model" : modelname}
        for tg in task_lingcat_scores_by_model[modelname].keys():
            unfair_tasks = []
            for qidx in task_lingcat_scores_by_model[modelname][tg].keys():
                question_id = int(qidx.split("_")[0])
                clip_llm_judge = True if question_id in pa.LLM_JUDGE_TASKS else False
                qid_min = pa.INDEX_MM_LOOKUP_TABLE[question_id]["min"]
                qid_max = pa.INDEX_MM_LOOKUP_TABLE[question_id]["max"]
    
                if tg == "conditioned_text_gen":
                    print(modelname, tg)
                    print(task_lingcat_scores_by_model[modelname][tg][qidx])
                this_task_unfair = False
                ttest_result = pa.compute_pairwise_t_test(
                    task_lingcat_scores_by_model[modelname][tg][qidx],
                    qid_min,
                    qid_max,
                    clip_llm_judge
                )
                for ttest_value in ttest_result.values():
                    if ttest_value < REJECT_THRESHOLD:
                        this_task_unfair = True
                unfair_tasks.append(this_task_unfair)
            row_instance[tg] = sum(unfair_tasks)/len(unfair_tasks) 
        rejectable_objlist.append(row_instance)
    rejectable_df = pd.DataFrame(rejectable_objlist)
    sortdcols = ["model"] + sorted([c for c in rejectable_df.columns if c != "model"])
    rejectable_df = rejectable_df[sortdcols]
    
    if CHAR_COLUMN == 0:
        rejectable_df.to_csv(f"score_tables/ptone_ptt.csv")
    elif CHAR_COLUMN == 2:
        rejectable_df.to_csv(f"score_tables/pgrammar_ptt.csv")
    else:
        if DO_ENCODE_MEDIAN:
            rejectable_df.to_csv(f"score_tables/p{FEAT_NAME}_ptt_median_feat.csv")
        else:
            rejectable_df.to_csv(f"score_tables/p{FEAT_NAME}_ptt.csv")

num_words 4


/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


gpt_4o conditioned_text_gen
{0: [0.96, 0.9425, 0.9475, 0.935, 0.9475, 0.965, 0.9525, 0.9375, 0.935, 0.9475, 0.96, 0.9425, 0.9425, 0.9575, 0.945, 0.94, 0.9625, 0.975, 0.9475], 1: [0.9675, 0.94, 0.9675, 0.955, 0.945, 0.945, 0.9575, 0.9675, 0.955, 0.98, 0.9625, 0.9625, 0.96, 0.9475, 0.95, 0.9375, 0.935, 0.94, 0.9475, 0.9575, 0.9525, 0.965, 0.9375, 0.9625, 0.94, 0.9375, 0.9425, 0.95, 0.9575, 0.9425, 0.9625, 0.9475, 0.9475, 0.94, 0.9625, 0.955, 0.955, 0.9475, 0.9625, 0.96, 0.95, 0.93, 0.9525, 0.98, 0.9475, 0.9425, 0.96, 0.9625, 0.9575, 0.945, 0.955, 0.95, 0.9575, 0.9475, 0.945, 0.95, 0.9575, 0.96, 0.955, 0.9725]}
llama31_70b_Inst conditioned_text_gen
{0: [0.935, 0.9175, 0.915, 0.8625, 0.9475, 0.9175, 0.9525, 0.9075, 0.95, 0.96, 0.96, 0.7925, 0.95, 0.9625, 0.9375, 0.9325, 0.9375, 0.8475, 0.955], 1: [0.96, 0.94, 0.94, 0.9675, 0.96, 0.9675, 0.94, 0.9275, 0.97, 0.865, 0.94, 0.9675, 0.9525, 0.9575, 0.9425, 0.9575, 0.9725, 0.88, 0.8925, 0.9375, 0.9475, 0.995, 0.9675, 0.9775, 0.9675, 0.89, 0.975, 

/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


gpt_4o conditioned_text_gen
{0: [0.96, 0.9425, 0.9675, 0.9475, 0.945, 0.935, 0.9475, 0.9675, 0.965, 0.9525, 0.9375, 0.96, 0.9475, 0.94, 0.9575, 0.9525, 0.94, 0.9375, 0.9575, 0.9625, 0.955, 0.955, 0.9625, 0.95, 0.9425, 0.9425, 0.9625, 0.945, 0.94, 0.9625, 0.95, 0.9575, 0.9475, 0.945, 0.9575, 0.96, 0.975], 1: [0.9675, 0.94, 0.955, 0.945, 0.9575, 0.955, 0.98, 0.9625, 0.9625, 0.935, 0.9475, 0.95, 0.9375, 0.935, 0.96, 0.9475, 0.965, 0.9375, 0.9625, 0.9425, 0.95, 0.9425, 0.9625, 0.9475, 0.9475, 0.94, 0.9475, 0.96, 0.93, 0.9525, 0.98, 0.9575, 0.9475, 0.9425, 0.96, 0.9575, 0.945, 0.955, 0.95, 0.955, 0.9475, 0.9725]}
llama31_70b_Inst conditioned_text_gen
{0: [0.935, 0.9175, 0.94, 0.915, 0.96, 0.8625, 0.9475, 0.9275, 0.9175, 0.9525, 0.9075, 0.9525, 0.96, 0.88, 0.9375, 0.9475, 0.9675, 0.89, 0.9675, 0.9825, 0.915, 0.9225, 0.955, 0.9025, 0.7925, 0.95, 0.9275, 0.9375, 0.9325, 0.9375, 0.945, 0.895, 0.9075, 0.925, 0.9275, 0.9075, 0.8475], 1: [0.96, 0.94, 0.9675, 0.9675, 0.94, 0.97, 0.865, 0.94, 0.9675

/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


... (4 outputs left out)

gpt_4o conditioned_text_gen
{1: [0.96, 0.9675, 0.94, 0.9425, 0.9675, 0.955, 0.9475, 0.945, 0.935, 0.9475, 0.945, 0.9575, 0.9675, 0.955, 0.98, 0.9625, 0.965, 0.9525, 0.9625, 0.9375, 0.96, 0.935, 0.9475, 0.9475, 0.95, 0.9375, 0.935, 0.96, 0.94, 0.9475, 0.9575, 0.9525, 0.965, 0.9375, 0.9625, 0.94, 0.9375, 0.9425, 0.95, 0.9575, 0.9425, 0.9625, 0.9475, 0.9475, 0.94, 0.9625, 0.955, 0.955, 0.9475, 0.9625, 0.96, 0.95, 0.93, 0.9425, 0.9525, 0.9425, 0.98, 0.9575, 0.9475, 0.9425, 0.96, 0.9625, 0.945, 0.94, 0.9575, 0.9625, 0.945, 0.955, 0.95, 0.9575, 0.9475, 0.945, 0.95, 0.9575, 0.96, 0.975, 0.955, 0.9475, 0.9725]}
llama31_70b_Inst conditioned_text_gen
{1: [0.935, 0.96, 0.94, 0.9175, 0.94, 0.9675, 0.915, 0.96, 0.8625, 0.9475, 0.9675, 0.94, 0.9275, 0.97, 0.865, 0.94, 0.9175, 0.9525, 0.9675, 0.9075, 0.9525, 0.95, 0.96, 0.9575, 0.9425, 0.9575, 0.9725, 0.96, 0.88, 0.8925, 0.9375, 0.9475, 0.995, 0.9675, 0.9775, 0.9675, 0.89, 0.975, 0.975, 0.9675, 0.97, 0.985, 0.9725, 0.9675, 0.96, 0.9825, 0.915, 0.9225

/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


gpt_4o conditioned_text_gen
{0: [0.96, 0.9425, 0.9675, 0.955, 0.9475, 0.945, 0.935, 0.9675, 0.9625, 0.9375, 0.9475, 0.9375, 0.94, 0.9575, 0.9525, 0.9375, 0.9575, 0.9625, 0.955, 0.9625, 0.95, 0.9425, 0.945, 0.94, 0.9575, 0.9475, 0.96, 0.975], 1: [0.9675, 0.94, 0.9475, 0.945, 0.9575, 0.955, 0.98, 0.9625, 0.965, 0.9525, 0.96, 0.935, 0.9475, 0.95, 0.935, 0.96, 0.9475, 0.965, 0.9375, 0.9625, 0.94, 0.9425, 0.95, 0.9425, 0.9625, 0.9475, 0.9475, 0.94, 0.955, 0.9475, 0.96, 0.93, 0.9525, 0.9425, 0.98, 0.9575, 0.9475, 0.9425, 0.96, 0.9625, 0.9575, 0.9625, 0.945, 0.955, 0.95, 0.945, 0.95, 0.9575, 0.955, 0.9475, 0.9725]}
llama31_70b_Inst conditioned_text_gen
{0: [0.935, 0.9175, 0.94, 0.9675, 0.915, 0.96, 0.8625, 0.9275, 0.9675, 0.9075, 0.96, 0.9575, 0.88, 0.9375, 0.9475, 0.89, 0.9675, 0.9825, 0.915, 0.955, 0.9025, 0.7925, 0.9375, 0.9325, 0.895, 0.9075, 0.9075, 0.8475], 1: [0.96, 0.94, 0.9475, 0.9675, 0.94, 0.97, 0.865, 0.94, 0.9175, 0.9525, 0.9525, 0.95, 0.9575, 0.9425, 0.9725, 0.96, 0.8925, 0.995,

/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/scipy/_lib/deprecation.py:234: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return f(*args, **kwargs)


# CORRELATION BETWEEN PROFILES

In [8]:
import numpy as np
from scipy.stats import chi2_contingency

def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

demo_profile_lists = {
    dp : [] for dp in pa.DEMOGRAPHY_PROFILES
}

for annfilename in os.listdir(pa.FINAL_ANNOTATOR_DIR):
    annotator_filepath = os.path.join(pa.FINAL_ANNOTATOR_DIR, annfilename)
    with open(annotator_filepath, "r") as ao:
        annotator_object = json.load(ao)
    if all(v is not None for v in annotator_object["profile"].values()):
        for dp, val in annotator_object["profile"].items():
            demo_profile_lists[dp].append(val)


dp_lists_categorical = pd.DataFrame(demo_profile_lists)
dpl_cat_cols = dp_lists_categorical.columns
n = len(dpl_cat_cols)

chi_matrix = pd.DataFrame(np.ones((n, n)), index=dpl_cat_cols, columns=dpl_cat_cols)
cramers_v_matrix = pd.DataFrame(np.ones((n, n)), index=dpl_cat_cols, columns=dpl_cat_cols)

for i in range(n):
    for j in range(n):
        if i == j:
            chi_matrix.iloc[i, j] = np.nan
            cramers_v_matrix.iloc[i, j] = 1.0
        else:
            contingency = pd.crosstab(dp_lists_categorical[dpl_cat_cols[i]], dp_lists_categorical[dpl_cat_cols[j]])
            chi2 = chi2_contingency(contingency)[0]
            
            chi_matrix.iloc[i, j] = chi2
            cramers_v_matrix.iloc[i, j] = cramers_v(contingency)

chi_matrix.to_csv(f"score_tables/demo_profiles_chi_matrix.csv")
cramers_v_matrix.to_csv(f"score_tables/demo_profiles_cramers_v_matrix.csv")


# SCATTER PLOT MATERIAL

In [9]:
scatter_plot_objlists = {}

for modl in pa.ALL_MODELS:
    scatter_plot_objlists[modl] = {}
    for k in scorelists_by_model[modl].keys():
        per_dp_objlist = []
        tasks = [qdi for qdi in scorelists_by_model[modl][k] if not any(qdi.startswith(prefix) for prefix in pa.EXCLUDED_SCORES)]
        for qdi in tasks:
            question_id = int(qdi.split("_")[0])
            clip_llm_judge = True if question_id in pa.LLM_JUDGE_TASKS else False
            qid_min = pa.INDEX_MM_LOOKUP_TABLE[question_id]["min"]
            qid_max = pa.INDEX_MM_LOOKUP_TABLE[question_id]["max"]
            all_values = sum(scorelists_by_model[modl][k][qdi].values(), [])
            all_values = pa.just_normalize(all_values, qid_min, qid_max, clip_llm_judge)
            divscore = pa.compute_diversity_score(scorelists_by_model[modl][k][qdi], qid_min, qid_max, clip_llm_judge)
            per_dp_objlist.append({
                "qdi" : qdi,
                "divscore" : divscore,
                "raw_mean" : sum(all_values)/len(all_values),
            })
        scatter_plot_objlists[modl][k] = per_dp_objlist
with open("score_tables/scatter_plot_objlists.json", "w+") as sfsf:
    sfsf.write(json.dumps(scatter_plot_objlists, indent=4))

In [10]:
print(scatter_plot_objlists.keys())
print(scatter_plot_objlists['gpt_4o'].keys())
print(json.dumps(scatter_plot_objlists['gpt_4o']['employment_sector'], indent=4))

dict_keys(['gpt_4o', 'llama31_70b_Inst', 'llama31_8B_Inst', 'llama32_1B_Inst', 'qwen25_72b_Inst', 'Qwen25_7B_Inst', 'Qwen25_1dot5B_Inst'])
dict_keys(['employment_sector', 'race', 'education', 'sex', 'age', 'language', 'nationality_group'])
[
    {
        "qdi": "0_0",
        "divscore": 0.029470149996025558,
        "raw_mean": 0.5805662238307487
    },
    {
        "qdi": "1_0",
        "divscore": 0.008556991969167186,
        "raw_mean": 0.9516781680894707
    },
    {
        "qdi": "2_11",
        "divscore": 0.008272050417688215,
        "raw_mean": 0.05156693499925273
    },
    {
        "qdi": "2_3",
        "divscore": 0.015910958890959764,
        "raw_mean": 0.731749940823243
    },
    {
        "qdi": "2_4",
        "divscore": 0.02941737710922268,
        "raw_mean": 0.07339735294992802
    },
    {
        "qdi": "2_10",
        "divscore": 0.01579429585358097,
        "raw_mean": 0.06874942666577646
    },
    {
        "qdi": "2_5",
        "divscore": 0.0088069892

# PRINTING TO LATEX

In [11]:
dvscores_path = "score_tables/divscores"
tgs = os.listdir(dvscores_path)
for tg in tgs:
    resdf = pd.read_csv(f"{dvscores_path}/{tg}", index_col=0)
    caption = tg.replace(".csv", "")
    print(pa.df_to_latex_one_column(resdf, caption, format="default"))
    print("=================================")

\begin{table*}[t]
\resizebox{\textwidth}{!}{%
\begin{tabular}{llllllll}
\toprule
model & industry & race & education & sex & age & language & nationality \\
\midrule
gpt\_4o & 0.0 & 0.0 & 0.01 & 0.01 & 0.0 & 0.0 & 0.01 \\
llama31\_70B\_Inst & 0.02 & 0.02 & 0.02 & 0.01 & 0.03 & 0.03 & 0.02 \\
llama31\_8B\_Inst & 0.06 & 0.03 & 0.05 & 0.06 & 0.08 & 0.03 & 0.04 \\
llama32\_1B\_Inst & 0.01 & 0.02 & 0.03 & 0.04 & 0.02 & 0.03 & 0.03 \\
Qwen25\_72B\_Inst & 0.01 & 0.02 & 0.02 & 0.01 & 0.02 & 0.01 & 0.01 \\
Qwen25\_7B\_Inst & 0.04 & 0.04 & 0.06 & 0.03 & 0.05 & 0.05 & 0.04 \\
Qwen25\_1dot5B\_Inst & 0.02 & 0.01 & 0.02 & 0.02 & 0.01 & 0.01 & 0.01 \\
\bottomrule
\end{tabular}
}
\caption{divscores\_conditioned\_text\_gen}
\label{tab:divscores_conditioned_text_gen}
\end{table*}

\begin{table*}[t]
\resizebox{\textwidth}{!}{%
\begin{tabular}{llllllll}
\toprule
model & industry & race & education & sex & age & language & nationality \\
\midrule
gpt\_4o & 0.04 & 0.04 & 0.05 & 0.03 & 0.03 & 0.04 & 0.04 \\


In [12]:
task_table = []
for i, q in enumerate(pa.QUESTIONS):
    for j in range(len(pa.QUESTIONS[q]["dataset"])):
        tc = f"{i}_{j}"
        task_obj = {
            # "qdi" : tc,
            "Group" : pa.QUESTIONS[q]["question_type"]
        }
        if not any(tc.startswith(prefix) for prefix in pa.EXCLUDED_SCORES):
            task_name = pa.QUESTIONS[q]["nickname"]
            dataset_name = pa.QUESTIONS[q]["dataset"][j]
            if "question_" not in dataset_name:
                task_name += f" - {dataset_name.replace('.csv', '')}"
                task_name = task_name.replace("500-","").replace("100-","").replace("grammatical_classification_","")
                # print(task_name)
            
            ds_df = pd.read_csv(f"question_artefacts/test_datasets/{dataset_name}")
            
            
            task_obj["Task Name and Benchmark"] = task_name
            task_obj["Dataset Size"] = len(ds_df)
            task_obj["Eval Metric"] = pa.QUESTIONS[q]["metrics"][0]
            
            task_table.append(task_obj)

task_table = pd.DataFrame(task_table)
print(task_table.to_latex(index=False, escape=True))

\begin{tabular}{llrl}
\toprule
Group & Task Name and Benchmark & Dataset Size & Eval Metric \\
\midrule
semantics & Sentiment Analysis 3 Classes - sentiment\_3class & 500 & f1 \\
semantics & Sentiment Analysis 2 Classes - sentiment\_2class & 500 & f1 \\
machine\_translation & Machine Translation - eng\_Latn-zho\_Hans & 100 & bleu \\
machine\_translation & Machine Translation - eng\_Latn-rus\_Cyrl & 100 & bleu \\
machine\_translation & Machine Translation - eng\_Latn-spa\_Latn & 100 & bleu \\
machine\_translation & Machine Translation - eng\_Latn-deu\_Latn & 100 & bleu \\
machine\_translation & Machine Translation - eng\_Latn-jpn\_Jpan & 100 & bleu \\
machine\_translation & Machine Translation - eng\_Latn-ind\_Latn & 100 & bleu \\
machine\_translation & Machine Translation - eng\_Latn-tgl\_Latn & 100 & bleu \\
machine\_translation & Machine Translation - eng\_Latn-hin\_Deva & 100 & bleu \\
machine\_translation & Machine Translation - eng\_Latn-arb\_Arab & 100 & bleu \\
machine\_translat

In [13]:
question_valid_ds = []
for q in pa.QUESTIONS:
    counter = 0
    for annfilename in os.listdir(pa.FINAL_ANNOTATOR_DIR):
        annotator_filepath = os.path.join(pa.FINAL_ANNOTATOR_DIR, annfilename)
        with open(annotator_filepath, "r") as  ao:
            annotator_object = json.load(ao)
        if annotator_object["answers"][q]["answer"]:
            counter += 1
    question_valid_ds.append({
        "Task" : pa.QUESTIONS[q]["nickname"],
        "Valid answers" : counter,
    })
        
question_valid_ds_df = pd.DataFrame(question_valid_ds).drop(pa.EXCLUDED_WHOLE)
print(question_valid_ds_df.to_latex(index=False, escape=True))


\begin{tabular}{lr}
\toprule
Task & Valid answers \\
\midrule
Sentiment Analysis 3 Classes & 80 \\
Sentiment Analysis 2 Classes & 90 \\
Machine Translation & 92 \\
Dialogue Summarization & 97 \\
Topic Classification & 90 \\
Free Generation: Balonku Song & 93 \\
Free Generation: G30SPKI & 93 \\
Free Generation: Hijrah & 80 \\
Free Generation: Adhan & 84 \\
Free Generation: Karak Kaliang & 94 \\
Free Generation: Giant Drum & 90 \\
Free Generation: Scientific Letter & 83 \\
Free Generation: School Life & 97 \\
Free Generation: Olympics Medallists & 96 \\
Free Generation: Hillary Clinton & 98 \\
Free Generation: MBZUAI & 98 \\
Free Generation: Grammar Correction & 98 \\
Free Generation: Grammar Correction CoT & 98 \\
Free Generation: Arithmetic & 99 \\
Free Generation: Shakespeare-Prompt Jailbreak & 97 \\
Multiple Choice Questions & 88 \\
Extractive QA & 91 \\
MultispanQA & 92 \\
Grammar Correctness Classification & 92 \\
Linguistically-conditioned Text Generation & 79 \\
\bottomrule
\end{

In [14]:
question_valid_ds_df["Valid answers"].sum()

2289

In [15]:
import pandas as pd

divsc_tab = pd.read_csv("score_tables/divscores/divscores_model_v_taskg.csv", index_col=0)
ptt_tab = pd.read_csv("score_tables/ptt/ptt_model_v_taskg.csv", index_col=0)
anova_tab = pd.read_csv("score_tables/anova/anova_model_v_taskg.csv", index_col=0)

In [16]:
print(pa.df_to_latex_one_column(divsc_tab, "CAPTION CAPTION CAPTION", format="percentage"))

\begin{table*}[t]
\resizebox{\textwidth}{!}{%
\begin{tabular}{llllllll}
\toprule
Model & semantics & machine\_translation & summarization & knowledge\_and\_reasoning & syntaxes & qa\_extraction & conditioned\_text\_gen \\
\midrule
gpt\_4o & 1.3\% & 1.1\% & 2.1\% & 3.8\% & 2.9\% & 2.0\% & 0.4\% \\
llama31\_70B\_Inst & 1.8\% & 0.8\% & 1.5\% & 5.6\% & 2.7\% & 3.0\% & 2.1\% \\
llama31\_8B\_Inst & 4.2\% & 0.5\% & 0.0\% & 6.1\% & 6.1\% & 2.7\% & 5.1\% \\
llama32\_1B\_Inst & 3.6\% & 0.4\% & 0.0\% & 6.3\% & 6.1\% & 4.4\% & 2.5\% \\
Qwen25\_72B\_Inst & 1.7\% & 0.6\% & 1.8\% & 5.1\% & 3.1\% & 2.6\% & 1.5\% \\
Qwen25\_7B\_Inst & 3.8\% & 0.7\% & 0.0\% & 6.9\% & 6.5\% & 2.3\% & 4.5\% \\
Qwen25\_1dot5B\_Inst & 4.1\% & 0.4\% & 0.0\% & 6.3\% & 7.7\% & 3.6\% & 1.6\% \\
\bottomrule
\end{tabular}
}
\caption{CAPTION CAPTION CAPTION}
\label{tab:CAPTION CAPTION CAPTION}
\end{table*}



In [17]:
print(pa.df_to_latex_one_column(ptt_tab, "PTT MAIN PTT MAIN PTT MAIN", format="percentage"))

\begin{table*}[t]
\resizebox{\textwidth}{!}{%
\begin{tabular}{llllllll}
\toprule
Model & semantics & machine\_translation & summarization & knowledge\_and\_reasoning & syntaxes & qa\_extraction & conditioned\_text\_gen \\
\midrule
gpt\_4o & 66.7\% & 64.3\% & 0.0\% & 61.5\% & 100.0\% & 33.3\% & 100.0\% \\
llama31\_70B\_Inst & 33.3\% & 64.3\% & 0.0\% & 61.5\% & 75.0\% & 66.7\% & 100.0\% \\
llama31\_8B\_Inst & 33.3\% & 71.4\% & 100.0\% & 46.2\% & 100.0\% & 100.0\% & 100.0\% \\
llama32\_1B\_Inst & 33.3\% & 35.7\% & 100.0\% & 30.8\% & 100.0\% & 33.3\% & 100.0\% \\
Qwen25\_72B\_Inst & 66.7\% & 50.0\% & 0.0\% & 30.8\% & 50.0\% & 66.7\% & 0.0\% \\
Qwen25\_7B\_Inst & 100.0\% & 35.7\% & 0.0\% & 46.2\% & 50.0\% & 0.0\% & 100.0\% \\
Qwen25\_1dot5B\_Inst & 66.7\% & 64.3\% & 0.0\% & 38.5\% & 50.0\% & 66.7\% & 0.0\% \\
\bottomrule
\end{tabular}
}
\caption{PTT MAIN PTT MAIN PTT MAIN}
\label{tab:PTT MAIN PTT MAIN PTT MAIN}
\end{table*}



In [18]:
print(pa.df_to_latex_one_column(anova_tab, "ANOVA MAIN ANOVA MAIN ANOVA MAIN", format="percentage"))

\begin{table*}[t]
\resizebox{\textwidth}{!}{%
\begin{tabular}{llllllll}
\toprule
Model & semantics & machine\_translation & summarization & knowledge\_and\_reasoning & syntaxes & qa\_extraction & conditioned\_text\_gen \\
\midrule
gpt\_4o & 33.3\% & 42.9\% & 0.0\% & 15.4\% & 0.0\% & 33.3\% & 0.0\% \\
llama31\_70B\_Inst & 33.3\% & 28.6\% & 0.0\% & 38.5\% & 50.0\% & 33.3\% & 100.0\% \\
llama31\_8B\_Inst & 33.3\% & 64.3\% & 100.0\% & 15.4\% & 75.0\% & 66.7\% & 100.0\% \\
llama32\_1B\_Inst & 0.0\% & 14.3\% & 100.0\% & 30.8\% & 75.0\% & 33.3\% & 0.0\% \\
Qwen25\_72B\_Inst & 66.7\% & 7.1\% & 0.0\% & 23.1\% & 50.0\% & 33.3\% & 0.0\% \\
Qwen25\_7B\_Inst & 100.0\% & 28.6\% & 0.0\% & 23.1\% & 50.0\% & 0.0\% & 100.0\% \\
Qwen25\_1dot5B\_Inst & 33.3\% & 35.7\% & 0.0\% & 23.1\% & 25.0\% & 33.3\% & 0.0\% \\
\bottomrule
\end{tabular}
}
\caption{ANOVA MAIN ANOVA MAIN ANOVA MAIN}
\label{tab:ANOVA MAIN ANOVA MAIN ANOVA MAIN}
\end{table*}



In [5]:
ptone_ptt = pd.read_csv("score_tables/ptone_ptt.csv", index_col=0)
pgrammar_ptt = pd.read_csv("score_tables/pgrammar_ptt.csv", index_col=0)
pgrammar_ptt = pd.read_csv("score_tables/pnumwords_ptt.csv", index_col=0)

# Convert the numeric cols to the desired string format
num_cols = ptone_ptt.columns.difference(['model'])
combined = ptone_ptt.copy()

combined[num_cols] = (
    (ptone_ptt[num_cols] * 100).round().astype(int).astype(str) + "% / " +
    (pgrammar_ptt[num_cols] * 100).round().astype(int).astype(str) + "%"
)

In [6]:
print(pa.df_to_latex_one_column(combined, "TONE AND GRAMMAR MAIN"))

\begin{table*}[t]
\resizebox{\textwidth}{!}{%
\begin{tabular}{llllllll}
\toprule
model & conditioned\_text\_gen & knowledge\_and\_reasoning & machine\_translation & qa\_extraction & semantics & summarization & syntaxes \\
\midrule
gpt\_4o & 0\% / 0\% & 23\% / 100\% & 86\% / 100\% & 33\% / 0\% & 33\% / 0\% & 0\% / 100\% & 0\% / 0\% \\
llama31\_70B\_Inst & 100\% / 0\% & 15\% / 100\% & 29\% / 100\% & 33\% / 0\% & 33\% / 0\% & 0\% / 100\% & 25\% / 0\% \\
llama31\_8B\_Inst & 100\% / 0\% & 8\% / 100\% & 71\% / 100\% & 0\% / 0\% & 67\% / 0\% & 0\% / 100\% & 75\% / 0\% \\
llama32\_1B\_Inst & 0\% / 0\% & 31\% / 100\% & 86\% / 100\% & 0\% / 0\% & 0\% / 0\% & 0\% / 100\% & 25\% / 0\% \\
Qwen25\_72B\_Inst & 100\% / 0\% & 38\% / 100\% & 57\% / 100\% & 33\% / 0\% & 33\% / 0\% & 0\% / 100\% & 25\% / 0\% \\
Qwen25\_7B\_Inst & 100\% / 0\% & 38\% / 100\% & 29\% / 100\% & 0\% / 0\% & 33\% / 0\% & 0\% / 100\% & 75\% / 0\% \\
Qwen25\_1dot5B\_Inst & 0\% / 0\% & 8\% / 100\% & 71\% / 100\% & 33\% / 0\% & 33\%

## Build table for t-test with new linguistic feats

In [7]:
ptone_ptt = pd.read_csv("score_tables/pnum_words_ptt.csv", index_col=0)
pgrammar_ptt = pd.read_csv("score_tables/pnum_short_words_ptt.csv", index_col=0)
pgrammar_ptt = pd.read_csv("score_tables/pnum_chars_ptt.csv", index_col=0)
p_tables = [
    pd.read_csv("score_tables/pnum_words_ptt.csv", index_col=0),
    pd.read_csv("score_tables/pnum_short_words_ptt.csv", index_col=0),
    pd.read_csv("score_tables/pnum_chars_ptt.csv", index_col=0),

    pd.read_csv("score_tables/pnum_stopwords_ptt.csv", index_col=0),
    pd.read_csv("score_tables/pnum_nouns_ptt.csv", index_col=0),
    #pd.read_csv("score_tables/pnum_pronouns_ptt.csv", index_col=0),
    #pd.read_csv("score_tables/pnum_exclamation_ptt.csv", index_col=0),
    #pd.read_csv("score_tables/pnum_long_words_ptt.csv", index_col=0),

    
]

# Convert the numeric cols to the desired string format
num_cols = p_tables[0].columns.difference(['model'])
combined = p_tables[0].copy()

combined[num_cols] = (
    (p_tables[0][num_cols] * 100).round().astype(int).astype(str) + "% / " +
    (p_tables[1][num_cols] * 100).round().astype(int).astype(str) + "% / " +
    (p_tables[2][num_cols] * 100).round().astype(int).astype(str) + "% / " +
    (p_tables[3][num_cols] * 100).round().astype(int).astype(str) + "% / " +
    (p_tables[4][num_cols] * 100).round().astype(int).astype(str) + "%"
    #(p_tables[5][num_cols] * 100).round().astype(int).astype(str) + "% / " +
    #(p_tables[6][num_cols] * 100).round().astype(int).astype(str) + "% / " +
    #(p_tables[7][num_cols] * 100).round().astype(int).astype(str) + "%"
)

In [8]:
print(pa.df_to_latex_one_column(combined, "TONE AND GRAMMAR MAIN"))

\begin{table*}[t]
\resizebox{\textwidth}{!}{%
\begin{tabular}{llllllll}
\toprule
model & conditioned\_text\_gen & knowledge\_and\_reasoning & machine\_translation & qa\_extraction & semantics & summarization & syntaxes \\
\midrule
gpt\_4o & 100\% / 100\% / 100\% / 0\% / 100\% & 85\% / 69\% / 77\% / 54\% / 46\% & 100\% / 100\% / 93\% / 21\% / 86\% & 100\% / 67\% / 67\% / 100\% / 33\% & 100\% / 100\% / 67\% / 100\% / 67\% & 100\% / 100\% / 100\% / 100\% / 100\% & 75\% / 100\% / 100\% / 75\% / 25\% \\
llama31\_70B\_Inst & 100\% / 100\% / 100\% / 100\% / 100\% & 62\% / 54\% / 46\% / 62\% / 31\% & 36\% / 100\% / 50\% / 50\% / 50\% & 67\% / 67\% / 67\% / 100\% / 33\% & 67\% / 100\% / 100\% / 33\% / 33\% & 100\% / 100\% / 100\% / 100\% / 0\% & 75\% / 50\% / 75\% / 100\% / 50\% \\
llama31\_8B\_Inst & 100\% / 100\% / 100\% / 100\% / 0\% & 77\% / 54\% / 69\% / 62\% / 54\% & 86\% / 93\% / 64\% / 50\% / 50\% & 100\% / 67\% / 100\% / 100\% / 33\% & 33\% / 100\% / 100\% / 67\% / 33\% & 100\% / 100\%

In [9]:
ptone_ptt = pd.read_csv("score_tables/pnum_words_ptt.csv", index_col=0)
pgrammar_ptt = pd.read_csv("score_tables/pnum_short_words_ptt.csv", index_col=0)
pgrammar_ptt = pd.read_csv("score_tables/pnum_chars_ptt.csv", index_col=0)
p_tables = [
    pd.read_csv("score_tables/pnum_words_ptt_median_feat.csv", index_col=0),
    pd.read_csv("score_tables/pnum_short_words_ptt_median_feat.csv", index_col=0),
    pd.read_csv("score_tables/pnum_chars_ptt_median_feat.csv", index_col=0),

    pd.read_csv("score_tables/pnum_stopwords_ptt_median_feat.csv", index_col=0),
    pd.read_csv("score_tables/pnum_nouns_ptt_median_feat.csv", index_col=0),
    #pd.read_csv("score_tables/pnum_pronouns_ptt.csv", index_col=0),
    #pd.read_csv("score_tables/pnum_exclamation_ptt.csv", index_col=0),
    #pd.read_csv("score_tables/pnum_long_words_ptt.csv", index_col=0),

    
]
# v2

p_tables = [
    pd.read_csv("score_tables/pratio_exclamation_ptt_median_feat_fixed_feats.csv", index_col=0),
    pd.read_csv("score_tables/pttr_ptt_median_feat_fixed_feats.csv", index_col=0),
    pd.read_csv("score_tables/pratio_nouns_ptt_median_feat_fixed_feats.csv", index_col=0),

    pd.read_csv("score_tables/pnum_words_ptt_median_feat_fixed_feats.csv", index_col=0),
    pd.read_csv("score_tables/psentence_type_ptt_median_feat_fixed_feats.csv", index_col=0),
    #pd.read_csv("score_tables/pnum_pronouns_ptt.csv", index_col=0),
    #pd.read_csv("score_tables/pnum_exclamation_ptt.csv", index_col=0),
    #pd.read_csv("score_tables/pnum_long_words_ptt.csv", index_col=0),

    
]
# Convert the numeric cols to the desired string format
num_cols = p_tables[0].columns.difference(['model'])
combined = p_tables[0].copy()

combined[num_cols] = (
    (p_tables[0][num_cols] * 100).round().astype(int).astype(str) + "% / " +
    (p_tables[1][num_cols] * 100).round().astype(int).astype(str) + "% / " +
    (p_tables[2][num_cols] * 100).round().astype(int).astype(str) + "% / " +
    (p_tables[3][num_cols] * 100).round().astype(int).astype(str) + "% / " +
    (p_tables[4][num_cols] * 100).round().astype(int).astype(str) + "%"
    #(p_tables[5][num_cols] * 100).round().astype(int).astype(str) + "% / " +
    #(p_tables[6][num_cols] * 100).round().astype(int).astype(str) + "% / " +
    #(p_tables[7][num_cols] * 100).round().astype(int).astype(str) + "%"
)

In [10]:
print(pa.df_to_latex_one_column(combined, "TONE AND GRAMMAR MAIN").replace("gpt\_4o", "GPT-4o").\
    replace("llama31\_70B\_Inst", "Llama3.1-70B-Instruct").\
    replace("llama31\_8B\_Inst", "Llama3.1-8B-Instruct").\
    replace("llama32\_1B\_Inst", "Llama3.2-1B-Instruct").\
    replace("Qwen25\_72B\_Inst", "Qwen2.5-72B-Instruct").\
    replace("Qwen25\_7B\_Inst", "Qwen2.5-7B-Instruct").\
    replace("Qwen25\_1dot5B\_Inst", "Qwen2.5-1.5B-Instruct"))

\begin{table*}[t]
\resizebox{\textwidth}{!}{%
\begin{tabular}{llllllll}
\toprule
model & conditioned\_text\_gen & knowledge\_and\_reasoning & machine\_translation & qa\_extraction & semantics & summarization & syntaxes \\
\midrule
GPT-4o & 0\% / 0\% / 0\% / 0\% / 0\% & 0\% / 8\% / 8\% / 15\% / 23\% & 0\% / 86\% / 57\% / 64\% / 86\% & 33\% / 33\% / 33\% / 0\% / 33\% & 0\% / 0\% / 33\% / 0\% / 33\% & 0\% / 0\% / 100\% / 100\% / 0\% & 0\% / 25\% / 0\% / 25\% / 0\% \\
Llama3.1-70B-Instruct & 0\% / 100\% / 100\% / 100\% / 100\% & 0\% / 15\% / 0\% / 0\% / 15\% & 0\% / 43\% / 29\% / 36\% / 29\% & 0\% / 0\% / 33\% / 33\% / 33\% & 0\% / 33\% / 0\% / 33\% / 33\% & 0\% / 0\% / 100\% / 100\% / 0\% & 0\% / 0\% / 0\% / 0\% / 25\% \\
Llama3.1-8B-Instruct & 0\% / 0\% / 0\% / 0\% / 100\% & 8\% / 8\% / 0\% / 8\% / 8\% & 0\% / 50\% / 36\% / 50\% / 71\% & 33\% / 0\% / 0\% / 0\% / 0\% & 33\% / 0\% / 67\% / 67\% / 67\% & 0\% / 0\% / 0\% / 100\% / 0\% & 25\% / 25\% / 50\% / 0\% / 75\% \\
Llama3.2-1B-Instruct

<>:1: SyntaxWarning: invalid escape sequence '\_'
<>:2: SyntaxWarning: invalid escape sequence '\_'
<>:3: SyntaxWarning: invalid escape sequence '\_'
<>:4: SyntaxWarning: invalid escape sequence '\_'
<>:5: SyntaxWarning: invalid escape sequence '\_'
<>:6: SyntaxWarning: invalid escape sequence '\_'
<>:7: SyntaxWarning: invalid escape sequence '\_'
<>:1: SyntaxWarning: invalid escape sequence '\_'
<>:2: SyntaxWarning: invalid escape sequence '\_'
<>:3: SyntaxWarning: invalid escape sequence '\_'
<>:4: SyntaxWarning: invalid escape sequence '\_'
<>:5: SyntaxWarning: invalid escape sequence '\_'
<>:6: SyntaxWarning: invalid escape sequence '\_'
<>:7: SyntaxWarning: invalid escape sequence '\_'
/tmp/ipykernel_1064252/3700571116.py:1: SyntaxWarning: invalid escape sequence '\_'
  print(pa.df_to_latex_one_column(combined, "TONE AND GRAMMAR MAIN").replace("gpt\_4o", "GPT-4o").\
/tmp/ipykernel_1064252/3700571116.py:2: SyntaxWarning: invalid escape sequence '\_'
  replace("llama31\_70B\_Inst", 

In [23]:
ptone_ptt = pd.read_csv("score_tables/pnum_words_ptt.csv", index_col=0)
pgrammar_ptt = pd.read_csv("score_tables/pnum_short_words_ptt.csv", index_col=0)
pgrammar_ptt = pd.read_csv("score_tables/pnum_chars_ptt.csv", index_col=0)
# ALL tasks 
# v2

p_tables = [
    pd.read_csv("score_tables/all_pratio_exclamation_ptt_median_feat_fixed_feats.csv", index_col=0),
    pd.read_csv("score_tables/all_pttr_ptt_median_feat_fixed_feats.csv", index_col=0),
    pd.read_csv("score_tables/all_pratio_nouns_ptt_median_feat_fixed_feats.csv", index_col=0),

    pd.read_csv("score_tables/all_pnum_words_ptt_median_feat_fixed_feats.csv", index_col=0),
    pd.read_csv("score_tables/all_psentence_type_ptt_median_feat_fixed_feats.csv", index_col=0),
    #pd.read_csv("score_tables/pnum_pronouns_ptt.csv", index_col=0),
    #pd.read_csv("score_tables/pnum_exclamation_ptt.csv", index_col=0),
    #pd.read_csv("score_tables/pnum_long_words_ptt.csv", index_col=0),

    
]
# Convert the numeric cols to the desired string format
num_cols = p_tables[0].columns.difference(['model'])
combined = p_tables[0].copy()

combined[num_cols] = (
    (p_tables[0][num_cols] * 100).round().astype(int).astype(str) + "% & " +
    (p_tables[1][num_cols] * 100).round().astype(int).astype(str) + "% & " +
    (p_tables[2][num_cols] * 100).round().astype(int).astype(str) + "% & " +
    (p_tables[3][num_cols] * 100).round().astype(int).astype(str) + "% & " +
    (p_tables[4][num_cols] * 100).round().astype(int).astype(str) + "%"
    #(p_tables[5][num_cols] * 100).round().astype(int).astype(str) + "% / " +
    #(p_tables[6][num_cols] * 100).round().astype(int).astype(str) + "% / " +
    #(p_tables[7][num_cols] * 100).round().astype(int).astype(str) + "%"
)

In [24]:
print(pa.df_to_latex_one_column(combined, "TONE AND GRAMMAR MAIN").replace("gpt\_4o", "GPT-4o").\
    replace("llama31\_70B\_Inst", "Llama3.1-70B-Instruct").\
    replace("llama31\_8B\_Inst", "Llama3.1-8B-Instruct").\
    replace("llama32\_1B\_Inst", "Llama3.2-1B-Instruct").\
    replace("Qwen25\_72B\_Inst", "Qwen2.5-72B-Instruct").\
    replace("Qwen25\_7B\_Inst", "Qwen2.5-7B-Instruct").\
    replace("Qwen25\_1dot5B\_Inst", "Qwen2.5-1.5B-Instruct"))

\begin{table*}[t]
\resizebox{\textwidth}{!}{%
\begin{tabular}{ll}
\toprule
model & machine\_translation \\
\midrule
GPT-4o & 3\% \& 38\% \& 31\% \& 33\% \& 44\% \\
Llama3.1-70B-Instruct & 0\% \& 26\% \& 18\% \& 23\% \& 26\% \\
Llama3.1-8B-Instruct & 10\% \& 23\% \& 23\% \& 28\% \& 44\% \\
Llama3.2-1B-Instruct & 8\% \& 33\% \& 26\% \& 31\% \& 44\% \\
Qwen2.5-72B-Instruct & 15\% \& 26\% \& 21\% \& 15\% \& 44\% \\
Qwen2.5-7B-Instruct & 10\% \& 13\% \& 28\% \& 23\% \& 36\% \\
Qwen2.5-1.5B-Instruct & 10\% \& 31\% \& 13\% \& 15\% \& 33\% \\
\bottomrule
\end{tabular}
}
\caption{TONE AND GRAMMAR MAIN}
\label{tab:TONE AND GRAMMAR MAIN}
\end{table*}



<>:1: SyntaxWarning: invalid escape sequence '\_'
<>:2: SyntaxWarning: invalid escape sequence '\_'
<>:3: SyntaxWarning: invalid escape sequence '\_'
<>:4: SyntaxWarning: invalid escape sequence '\_'
<>:5: SyntaxWarning: invalid escape sequence '\_'
<>:6: SyntaxWarning: invalid escape sequence '\_'
<>:7: SyntaxWarning: invalid escape sequence '\_'
<>:1: SyntaxWarning: invalid escape sequence '\_'
<>:2: SyntaxWarning: invalid escape sequence '\_'
<>:3: SyntaxWarning: invalid escape sequence '\_'
<>:4: SyntaxWarning: invalid escape sequence '\_'
<>:5: SyntaxWarning: invalid escape sequence '\_'
<>:6: SyntaxWarning: invalid escape sequence '\_'
<>:7: SyntaxWarning: invalid escape sequence '\_'
/tmp/ipykernel_1064252/3700571116.py:1: SyntaxWarning: invalid escape sequence '\_'
  print(pa.df_to_latex_one_column(combined, "TONE AND GRAMMAR MAIN").replace("gpt\_4o", "GPT-4o").\
/tmp/ipykernel_1064252/3700571116.py:2: SyntaxWarning: invalid escape sequence '\_'
  replace("llama31\_70B\_Inst", 

In [7]:
import pandas as pd

asd1 = pd.read_csv("score_tables/demo_profiles_chi_matrix.csv", index_col=0)
asd1

,employment_sector,race,education,sex,age,language,nationality_group
employment_sector,NaN,10.076040,2.060739,5.818907,11.152491,10.942798,12.162088
race,10.076040,NaN,12.988852,8.301446,35.234962,47.142857,65.563636
education,2.060739,12.988852,NaN,0.168644,3.329893,4.368503,19.958914
sex,5.818907,8.301446,0.168644,NaN,3.131976,4.261798,7.020503
age,11.152491,35.234962,3.329893,3.131976,NaN,40.595238,45.401316
language,10.942798,47.142857,4.368503,4.261798,40.595238,NaN,49.814286
nationality_group,12.162088,65.563636,19.958914,7.020503,45.401316,49.814286,NaN


In [8]:
asd2 = pd.read_csv("score_tables/demo_profiles_cramers_v_matrix.csv", index_col=0)
asd2

,employment_sector,race,education,sex,age,language,nationality_group
employment_sector,1.000000,0.302656,0.136872,0.325267,0.318412,0.446049,0.332512
race,0.302656,1.000000,0.343628,0.388504,0.565966,0.925820,0.772032
education,0.136872,0.343628,1.000000,0.055374,0.173988,0.281829,0.425963
sex,0.325267,0.388504,0.055374,1.000000,0.238632,0.278365,0.357275
age,0.318412,0.565966,0.173988,0.238632,1.000000,0.859125,0.642448
language,0.446049,0.925820,0.281829,0.278365,0.859125,1.000000,0.951690
nationality_group,0.332512,0.772032,0.425963,0.357275,0.642448,0.951690,1.000000
